In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("global air pollution dataset.csv")
df.head()

,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good
2,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate
3,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good
4,France,Punaauia,22,Good,0,Good,22,Good,0,Good,6,Good


In [4]:
df.columns

Index(['Country', 'City', 'AQI Value', 'AQI Category', 'CO AQI Value',
       'CO AQI Category', 'Ozone AQI Value', 'Ozone AQI Category',
       'NO2 AQI Value', 'NO2 AQI Category', 'PM2.5 AQI Value',
       'PM2.5 AQI Category'],
      dtype='object')

In [5]:
df = df.rename(columns={
    'AQI Value': 'aqi',
    'CO AQI Value': 'co',
    'Ozone AQI Value': 'ozone',
    'NO2 AQI Value': 'no2',
    'PM2.5 AQI Value': 'pm25'
})

df.head()

,Country,City,aqi,AQI Category,co,CO AQI Category,ozone,Ozone AQI Category,no2,NO2 AQI Category,pm25,PM2.5 AQI Category
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good
2,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate
3,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good
4,France,Punaauia,22,Good,0,Good,22,Good,0,Good,6,Good


In [6]:
df['stress_index'] = (
    df['aqi'] * 0.4 +
    df['pm25'] * 0.3 +
    df['no2'] * 0.15 +
    df['ozone'] * 0.1 +
    df['co'] * 0.05
)

df[['Country', 'City', 'stress_index']].head()

,Country,City,stress_index
0,Russian Federation,Praskoveya,39.35
1,Brazil,Presidente Dutra,29.40
2,Italy,Priolo Gargallo,50.45
3,Poland,Przasnysz,23.05
4,France,Punaauia,12.80


In [7]:
df.isnull().sum() # sanity check

Country               427
City                    1
aqi                     0
AQI Category            0
co                      0
CO AQI Category         0
ozone                   0
Ozone AQI Category      0
no2                     0
NO2 AQI Category        0
pm25                    0
PM2.5 AQI Category      0
stress_index            0
dtype: int64

In [8]:
df.to_csv("clean_environment_data.csv", index=False)

Below contains the implementation of the SQL dashboard

In [ ]:
import sqlite3 

conn = sqlite3.connect("environment.db")
df.to_sql("environmental_data", conn, if_exists="replace", index=False)

print("Saved table: environmental_data")

Saved table: environmental_data


In [ ]:
pd.read_sql("SELECT * FROM environmental_data LIMIT 5;", conn) # test tables

,Country,City,aqi,AQI Category,co,CO AQI Category,ozone,Ozone AQI Category,no2,NO2 AQI Category,pm25,PM2.5 AQI Category,stress_index
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate,39.35
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good,29.40
2,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate,50.45
3,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good,23.05
4,France,Punaauia,22,Good,0,Good,22,Good,0,Good,6,Good,12.80


In [ ]:
#query for countries with highest avg of environmental stress

pd.read_sql("""
SELECT Country,
       ROUND(AVG(stress_index), 2) AS avg_stress_index
FROM environmental_data
GROUP BY Country
ORDER BY avg_stress_index DESC
LIMIT 10;
""", conn)


,Country,avg_stress_index
0,Republic of Korea,307.90
1,Bahrain,144.85
2,Pakistan,132.78
3,Kuwait,129.15
4,Mauritania,128.19
5,United Arab Emirates,127.80
6,Qatar,121.43
7,Aruba,116.40
8,Saudi Arabia,112.37
9,India,111.91


In [ ]:
#query for cities with highest avg of AQI


pd.read_sql("""
SELECT City, Country,
       ROUND(AVG(aqi), 2) AS avg_aqi
FROM environmental_data
GROUP BY City, Country
ORDER BY avg_aqi DESC
LIMIT 10;
""", conn)

,City,Country,avg_aqi
0,Aonla,India,500.0
1,Bagar,India,500.0
2,Bahawalnagar,Pakistan,500.0
3,Bahjoi,India,500.0
4,Balotra,India,500.0
5,Bareli,India,500.0
6,Barkhera,India,500.0
7,Bawal,India,500.0
8,Bhadasar,India,500.0
9,Bhadra,India,500.0


In [ ]:
#query on avg polluntant level by country

pd.read_sql("""
SELECT Country,
       ROUND(AVG(pm25), 2) AS avg_pm25,
       ROUND(AVG(no2), 2) AS avg_no2,
       ROUND(AVG(ozone), 2) AS avg_ozone,
       ROUND(AVG(co), 2) AS avg_co
FROM environmental_data
GROUP BY Country
ORDER BY avg_pm25 DESC
LIMIT 10;
""", conn)

,Country,avg_pm25,avg_no2,avg_ozone,avg_co
0,Republic of Korea,415.00,91.00,0.00,27.00
1,Bahrain,188.00,3.00,127.00,2.00
2,Mauritania,179.00,0.00,28.50,0.75
3,Pakistan,173.11,2.07,89.14,2.07
4,Aruba,163.00,0.00,23.00,0.00
5,Kuwait,162.00,13.67,135.67,2.67
6,United Arab Emirates,152.67,3.33,159.67,1.33
7,Senegal,152.42,1.88,22.00,1.39
8,India,149.46,1.97,55.06,1.74
9,Saudi Arabia,149.29,0.64,77.07,1.29


In [14]:
#query for air quality distribution categories

pd.read_sql("""
SELECT [AQI Category],
       COUNT(*) AS record_count
FROM environmental_data
GROUP BY [AQI Category]
ORDER BY record_count DESC;
""", conn)

,AQI Category,record_count
0,Good,9936
1,Moderate,9231
2,Unhealthy,2227
3,Unhealthy for Sensitive Groups,1591
4,Very Unhealthy,287
5,Hazardous,191


In [17]:
#highest stress cities within each country

pd.read_sql("""
SELECT Country, City,
       ROUND(stress_index, 2) AS stress_index
FROM environmental_data
ORDER BY stress_index DESC
LIMIT 15;
""", conn)


,Country,City,stress_index
0,United States of America,Durango,364.60
1,India,Aonla,359.40
2,India,Gunnaur,357.50
3,India,Puranpur,357.35
4,India,Barkhera,357.25
5,India,Moradabad,357.10
6,South Africa,Boksburg,357.05
7,India,Tajpur,356.90
8,India,Kandhla,356.65
9,India,Gulaothi,356.55


In [19]:
#country-level summary

country_summary = pd.read_sql("""
SELECT Country,
       ROUND(AVG(aqi), 2) AS avg_aqi,
       ROUND(AVG(pm25), 2) AS avg_pm25,
       ROUND(AVG(no2), 2) AS avg_no2,
       ROUND(AVG(ozone), 2) AS avg_ozone,
       ROUND(AVG(co), 2) AS avg_co,
       ROUND(AVG(stress_index), 2) AS avg_stress_index,
       COUNT(DISTINCT City) AS city_count
FROM environmental_data
GROUP BY Country
ORDER BY avg_stress_index DESC;
""", conn)

country_summary.to_csv("country_summary.csv", index=False)
country_summary.head()


,Country,avg_aqi,avg_pm25,avg_no2,avg_ozone,avg_co,avg_stress_index,city_count
0,Republic of Korea,421.00,415.00,91.00,0.00,27.00,307.90,1
1,Bahrain,188.00,188.00,3.00,127.00,2.00,144.85,1
2,Pakistan,178.79,173.11,2.07,89.14,2.07,132.78,307
3,Kuwait,162.00,162.00,13.67,135.67,2.67,129.15,3
4,Mauritania,179.00,179.00,0.00,28.50,0.75,128.19,4
